# 04 - Model training

Purpose: run the reproducible training pipeline and inspect what it did.

The implementation lives in `rainml.training.trainer.run_training`; the same function
is invoked by `scripts/run_training.py` and by the retraining job. This notebook calls
it and reports the result, so nothing here can drift from production.

## Stages

1. load observations and district coordinates
2. validate the dataset (hard failure on invalid data)
3. clean impossible values, impute missing days
4. engineer features and check for leakage
5. chronological split
6. fit four baselines
7. fit four candidates
8. score on validation, select the best candidate
9. rescore on the untouched test slice
10. fit quantile models for the forecast interval
11. write the versioned artifact and model card

## Reproducibility

`random_seed` is fixed in `ml/configs/training.yaml`. Re-running with the same
dataset and seed produces the same metrics.

In [ ]:
import sys
from pathlib import Path

# The notebook kernel runs with the notebook directory as the working directory,
# but paths are resolved from the repository root so cells behave the same when
# executed by nbconvert, JupyterLab or an IDE.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "ml" / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

ML_SRC = REPO_ROOT / "ml" / "src"
if str(ML_SRC) not in sys.path:
    sys.path.insert(0, str(ML_SRC))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
plt.style.use("seaborn-v0_8-whitegrid")

DATA_DIR = REPO_ROOT / "data" / "processed"
ARTIFACT_ROOT = REPO_ROOT / "ml" / "artifacts"
CONFIG_PATH = REPO_ROOT / "ml" / "configs" / "training.yaml"
OBSERVATIONS = DATA_DIR / "observations.csv"
DISTRICTS = DATA_DIR / "districts.csv"

print("notebook environment ready")
print("repo root:", REPO_ROOT)
print("python", sys.version.split()[0])


In [ ]:
from rainml.training.configuration import load_config, TrainingConfig
from rainml.training.trainer import run_training

config = load_config(CONFIG_PATH)
config.random_seed = 42
config.fit_interval_models = True
config

In [ ]:
report = run_training(
    observations_path=OBSERVATIONS,
    districts_path=DISTRICTS,
    artifact_root=ARTIFACT_ROOT,
    config=config,
    progress=print,
)
print(report.status, report.error)

## Dataset and split

The training slice must end before the validation slice, which must end before the
test slice. `validate_split` enforces this inside the pipeline.

In [ ]:
pd.DataFrame(
    [
        {"split": "train", "rows": report.split_sizes["train"], "first": report.split_dates["train_end"]},
        {"split": "valid", "rows": report.split_sizes["valid"], "first": report.split_dates["valid_start"]},
        {"split": "test", "rows": report.split_sizes["test"], "first": report.split_dates["test_start"]},
    ]
)

## Baseline versus candidate

A candidate is only worth serving if it beats the best baseline. `beats_baseline` is
the gate the promotion policy later uses.

In [ ]:
validation_scores = report.metrics["validation"]
scores = pd.DataFrame(validation_scores).T[["mae_mm", "rmse_mm", "bias_mm", "csi", "brier_score"]]
scores.sort_values("mae_mm").round(4)

In [ ]:
print("best baseline :", report.comparison["best_baseline"])
print("best candidate:", report.comparison["best_candidate"])
print("skill vs baseline:", report.comparison["skill_vs_best_baseline"])
print("beats baseline:", report.comparison["beats_baseline"])

## Test slice

These are the numbers quoted as the model's real skill: the test partition was not
used for feature selection, hyperparameter choice or model choice.

In [ ]:
pd.DataFrame(report.metrics["test"]).T[["mae_mm", "rmse_mm", "bias_mm", "nash_sutcliffe", "sample_size"]].round(4)

## Selected model and artifact

In [ ]:
print("selected:", report.selected_model)
print("version :", report.version)
print("artifact:", report.artifact_path)
print("features:", report.feature_selection["selected"])

In [ ]:
import json

metadata = json.loads((Path(report.artifact_path) / "metadata.json").read_text())
pd.Series({k: metadata[k] for k in ["model_name", "version", "trained_at", "training_data_range", "framework_version", "status"]})

## Reproducibility check

Training twice with the same seed must give the same validation MAE.

In [ ]:
import tempfile

scratch = Path(tempfile.mkdtemp(prefix="rainml_repro_"))
repeat = run_training(
    observations_path=OBSERVATIONS,
    districts_path=DISTRICTS,
    artifact_root=scratch,
    config=config,
)
print("first :", report.metrics["validation"][report.selected_model]["mae_mm"])
print("repeat:", repeat.metrics["validation"][repeat.selected_model]["mae_mm"])